# RR BENCHMARK PHASE A: library, drug IDs, series survey (NO SCORING)
Pre-registration committed 2026-10-01 15:47 EDT, SHA-256 7eee1779...c9a2.
This notebook computes **no connectivity score**. It builds the full-landmark library, resolves the drug ID sets, and surveys the 8 series.
Run: hamburger (top left) > Runtime > Run all. Then copy the text between `=== PHASE A REPORT BEGIN ===` and `=== PHASE A REPORT END ===` and paste it back.

In [ ]:
# CELL 1: mount Drive, work folder
from google.colab import drive
drive.mount("/content/drive")
import os,subprocess,sys
W="/content/drive/MyDrive/RitschelResearch/benchmark"; os.makedirs(W,exist_ok=True); os.chdir(W)
subprocess.run([sys.executable,"-m","pip","install","-q","fsspec","aiohttp","h5py"],check=False)
print("workdir",os.getcwd()); print(sorted(os.listdir(W))[:30])

In [ ]:
# CELL 2: LINCS 2020 metadata
import urllib.request,os,pandas as pd
B="https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/"
for f in ["siginfo_beta.txt","geneinfo_beta.txt","compoundinfo_beta.txt"]:
    if not os.path.exists(f): urllib.request.urlretrieve(B+f,f)
    print(f,os.path.getsize(f))
g=pd.read_csv("geneinfo_beta.txt",sep="\t")
LM=g.loc[g.feature_space=="landmark",["gene_id","gene_symbol"]].copy(); LM["gene_id"]=LM.gene_id.astype(int)
print("landmark genes:",len(LM))

In [ ]:
# CELL 3: full-landmark library (QC-pass exemplar trt_cp), same filter as control 2, ALL landmarks.
# Skipped if already on Drive. Remote slice of the 35 GB gctx; expect 20-60 min.
import time,numpy as np,pandas as pd,fsspec,h5py,os
if os.path.exists("libA_matrix.npy") and os.path.exists("libA_perts.npy") and os.path.exists("libA_genes.npy"):
    print("library already built, skipping")
else:
    t0=time.time(); lm=set(LM.gene_id)
    s=pd.read_csv("siginfo_beta.txt",sep="\t",usecols=["sig_id","pert_id","pert_type","is_exemplar_sig","qc_pass"],low_memory=False)
    s=s[(s.pert_type=="trt_cp")&(s.is_exemplar_sig==1)&(s.qc_pass==1)]
    print("qc exemplar trt_cp signatures:",len(s),"compounds:",s.pert_id.nunique())
    u="https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/level5/level5_beta_trt_cp_n720216x12328.gctx"
    h=h5py.File(fsspec.open(u,"rb").open(),"r"); d=h["0"]["DATA"]["0"]["matrix"]
    colid=np.array([x.decode() for x in h["0"]["META"]["COL"]["id"][:]])
    rowid=np.array([int(x.decode()) for x in h["0"]["META"]["ROW"]["id"][:]])
    assert d.shape[0]==len(colid) and d.shape[1]==len(rowid), "AXIS LABELS NOT AS EXPECTED"
    gcol=np.array([i for i,x in enumerate(rowid) if x in lm]); print("landmark columns:",len(gcol))
    pos={x:i for i,x in enumerate(colid)}
    want=sorted((pos[x],p) for x,p in zip(s.sig_id,s.pert_id) if x in pos)
    idx=np.array([w[0] for w in want]); perts=np.array([w[1] for w in want])
    out=np.empty((len(idx),len(gcol)),dtype=np.float32); t1=time.time()
    for a in range(0,len(idx),2000):
        b=min(a+2000,len(idx)); out[a:b]=d[idx[a:b],:][:,gcol]
        if (a//2000)%5==0: print("  %6d/%6d %.0fs"%(b,len(idx),time.time()-t1),flush=True)
    np.save("libA_matrix.npy",out); np.save("libA_perts.npy",perts); np.save("libA_genes.npy",rowid[gcol])
    print("saved",out.shape,"finite",float(np.isfinite(out).mean()),"TOTAL",round(time.time()-t0),"s")
M=np.load("libA_matrix.npy",mmap_mode="r"); perts=np.load("libA_perts.npy",allow_pickle=True); genes=np.load("libA_genes.npy")
# read-back check: a write is not done until a read confirms it
assert M.shape[0]==len(perts) and M.shape[1]==len(genes)==len(set(genes)), "LIBRARY READ-BACK MISMATCH"
LIB=dict(sigs=int(M.shape[0]),genes=int(M.shape[1]),compounds=int(len(set(perts.tolist()))))
print("LIBRARY",LIB)

In [ ]:
# CELL 4: drug ID resolution BY RULE (g2_decisions.csv was lost). Printed for GR to confirm BEFORE phase B.
import pandas as pd,numpy as np,collections
ci=pd.read_csv("compoundinfo_beta.txt",sep="\t",dtype=str).fillna("")
si=pd.read_csv("siginfo_beta.txt",sep="\t",usecols=["sig_id","pert_id","pert_type","is_exemplar_sig","qc_pass","cell_iname"],low_memory=False)
si=si[(si.pert_type=="trt_cp")&(si.is_exemplar_sig==1)&(si.qc_pass==1)]
inlib=collections.Counter(perts.tolist())
DRUGS={6:["cladribine"],8:["tadalafil"],10:["doxycycline"],12:["hydroxyurea"],13:["ruxolitinib"],
       14:["dapagliflozin","empagliflozin","spironolactone"],17:["cyclosporin a","cyclosporin-a","cyclosporine","ciclosporin"],18:["tacrolimus"]}
EXCLUDE={"BRD-K19601669"}   # S-ruxolitinib, excluded in the protocol
def hits(name):
    n=name.lower()
    m=ci[(ci.cmap_name.str.lower()==n)|(ci.compound_aliases.str.lower().str.split("|").apply(lambda L: n in [x.strip() for y in L for x in y.split(",")]))]
    return m
rows=[]
for r,names in DRUGS.items():
    for nm in names:
        for _,x in hits(nm).iterrows():
            rows.append(dict(row=r,query_name=nm,pert_id=x.pert_id,cmap_name=x.cmap_name,inchi_key=x.inchi_key,
                             skeleton=x.inchi_key[:14],excluded=x.pert_id in EXCLUDE,
                             n_lib_sigs=int(inlib.get(x.pert_id,0)),
                             n_cells=int(si[si.pert_id==x.pert_id].cell_iname.nunique())))
# doxycycline: protocol says the CMap rows carry skeleton VXVNYCRICCVPTD (amide tautomer + stereo variant); add any by skeleton
dx=ci[ci.inchi_key.str[:14]=="VXVNYCRICCVPTD"]
for _,x in dx.iterrows():
    if not any(q["pert_id"]==x.pert_id and q["row"]==10 for q in rows):
        rows.append(dict(row=10,query_name="doxycycline(skeleton)",pert_id=x.pert_id,cmap_name=x.cmap_name,inchi_key=x.inchi_key,
                         skeleton=x.inchi_key[:14],excluded=False,n_lib_sigs=int(inlib.get(x.pert_id,0)),
                         n_cells=int(si[si.pert_id==x.pert_id].cell_iname.nunique())))
D=pd.DataFrame(rows).drop_duplicates(["row","pert_id"]).sort_values(["row","query_name","pert_id"])
D.to_csv("drugs_resolved_phaseA.csv",index=False)
print(D.to_string(index=False))

In [ ]:
# CELL 5: series survey. Matrix table present? platform, sample count, supplementary files, landmark coverage.
import gzip,urllib.request,os,re,collections
SER=[("6","GSE174647",None),("8","GSE256539",None),("10","GSE10334",None),("12","GSE35007",None),
     ("13","GSE134662",None),("14/15","GSE57338",None),("17","GSE6710",None),("18","GSE16161","GPL570")]
def get_matrix(g,p):
    f=(g+"-"+p if p else g)+"_series_matrix.txt"
    if not os.path.exists(f):
        s=g[:-3]+"nnn"; name=(g+"-"+p if p else g)+"_series_matrix.txt.gz"
        u="https://ftp.ncbi.nlm.nih.gov/geo/series/%s/%s/matrix/%s"%(s,g,name)
        open(f,"wb").write(gzip.decompress(urllib.request.urlopen(u,timeout=120).read()))
    return f
def get_annot(plat):
    f=plat+".annot.gz"
    if not os.path.exists(f):
        u="https://ftp.ncbi.nlm.nih.gov/geo/platforms/%snnn/%s/annot/%s.annot.gz"%(plat[:-3],plat,plat)
        try: urllib.request.urlretrieve(u,f)
        except Exception as e: return None
    return f
lmids=set(LM.gene_id); REP=[]
for row,g,p in SER:
    try: f=get_matrix(g,p)
    except Exception as e: REP.append(f"row {row} {g}: MATRIX FETCH FAILED {e}"); continue
    hdr=[];nrow=0;inb=False;first=None
    for line in open(f,encoding="utf-8",errors="replace"):
        if line.startswith("!series_matrix_table_end"): break
        if inb:
            nrow+=1
            if nrow==2: first=line[:80].strip()
        else: hdr.append(line)
        if line.startswith("!series_matrix_table_begin"): inb=True
    H="".join(hdr)
    plat=re.findall(r'!Series_platform_id\t"?([^"\t\n]+)',H)
    m=re.search(r"!Sample_geo_accession(.*)",H); ns=len(re.findall(r"GSM\d+",m.group(1))) if m else 0
    sup=re.findall(r'!Series_supplementary_file\t(.*)',H); sup=[x.strip().strip('"') for l in sup for x in l.split("\t")]
    lt=set(re.findall(r'!Sample_library_strategy\t(.*)',H)[:1])
    nprobe=max(nrow-1,0)
    cov="n/a"
    if nprobe>0 and plat:
        a=get_annot(plat[0])
        if a:
            ids=set(); inb=False; ige=None
            for raw in gzip.open(a,"rt",encoding="utf-8",errors="replace"):
                if raw.startswith("!platform_table_end"): break
                if inb:
                    q=raw.rstrip("\n").split("\t")
                    if ige is None: ige=[i for i,c in enumerate(q) if "Gene ID" in c]; continue
                    if ige and len(q)>ige[0]:
                        x=q[ige[0]].split("///")[0].strip()
                        if x.isdigit(): ids.add(int(x))
                if raw.startswith("!platform_table_begin"): inb=True
            cov="%d of %d landmarks"%(len(ids&lmids),len(lmids))
        else: cov="annot unavailable"
    REP.append(f"row {row} {g}{'-'+p if p else ''}: platform {plat} samples {ns} matrix_rows {nprobe} first_row {first!r} lib_strategy {lt or '-'} landmark_cov {cov}")
    REP.append("   supplementary: "+(" | ".join(sup) if sup else "NONE"))
print("\n".join(REP))

In [ ]:
# CELL 6: report
print("=== PHASE A REPORT BEGIN ===")
print("LIBRARY",LIB)
print("--- DRUGS (resolved by rule; confirm before phase B) ---")
print(D[["row","query_name","pert_id","cmap_name","skeleton","excluded","n_lib_sigs","n_cells"]].to_string(index=False))
print("--- SERIES ---"); print("\n".join(REP))
print("=== PHASE A REPORT END ===")